# 20BN-Jester 資料分析（階段 1）

目的：從 CSV 與影格檔了解資料，產出後續設計要用的三個決策：

1. **取樣幾張影格 `T`** —— 看影格數分佈
2. **Resize 成多大** —— 看影像尺寸分佈
3. **要不要處理類別不平衡** —— 看 27 類各有多少樣本

外加一步肉眼檢查：隨機抽幾段影片，把影格排出來確認手勢對得上標籤。

> 註：本版（toxicmender）的 `frames`、`shape` 已預先寫在 CSV，前三項幾乎只是「讀 CSV → 畫圖」。

## 0. 設定與匯入

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# 資料根目錄（之後寫程式時建議搬到 config.py）
DATA_ROOT = Path(r"D:/JESTER/data")
TRAIN_DIR = DATA_ROOT / "Train"
VAL_DIR   = DATA_ROOT / "Validation"
TEST_DIR  = DATA_ROOT / "Test"

print("DATA_ROOT 存在:", DATA_ROOT.exists())
for d in (TRAIN_DIR, VAL_DIR, TEST_DIR):
    print(f"  {d.name:11s} 存在:", d.exists())

## 1. 讀取三個 CSV

本版 CSV 是**逗號分隔、有表頭**，直接 `pd.read_csv()` 即可。
欄位：`video_id`(Test 為 `id`)、`label`、`frames`、`label_id`、`shape`、`format`。

In [ ]:
train_df = pd.read_csv(DATA_ROOT / "Train.csv")
val_df   = pd.read_csv(DATA_ROOT / "Validation.csv")
test_df  = pd.read_csv(DATA_ROOT / "Test.csv")

print("Train     :", train_df.shape)
print("Validation:", val_df.shape)
print("Test      :", test_df.shape)
train_df.head()

In [ ]:
# Test.csv 的 label / label_id 是空的（測試集無標籤）
test_df.head()

## 2. 影格數分佈（frames）→ 決定取樣張數 `T`

In [ ]:
print("Train frames 統計:")
print(train_df["frames"].describe())
print()
print("Train frames 不同值:", sorted(train_df["frames"].unique()))
print("Val   frames 不同值:", sorted(val_df["frames"].unique()))
# 預期：本子集全部固定 37 幀

## 3. 影像尺寸分佈（shape）→ 決定 resize 尺寸

In [ ]:
size_counts = train_df["shape"].value_counts()
print(size_counts)

size_counts.plot(kind="bar")
plt.title("Frame shape distribution (Train)")
plt.xlabel("(height, width)")
plt.ylabel("count")
plt.tight_layout()
plt.show()

## 4. 類別分佈 → 決定是否處理不平衡

In [ ]:
cls = (train_df.groupby(["label_id", "label"]).size()
       .reset_index(name="count").sort_values("label_id"))
cls

In [ ]:
plt.figure(figsize=(9, 7))
plt.barh(cls["label"], cls["count"])
plt.gca().invert_yaxis()
plt.title("Class distribution (Train)")
plt.xlabel("count")
plt.tight_layout()
plt.show()

print("最多:", cls.loc[cls['count'].idxmax(), 'label'], int(cls['count'].max()))
print("最少:", cls.loc[cls['count'].idxmin(), 'label'], int(cls['count'].min()))
print("不平衡比 (max/min):", round(cls['count'].max() / cls['count'].min(), 2))

## 5. 隨機抽影片看影格

把一段影片均勻取樣幾張影格排成一列，肉眼確認手勢內容對得上標籤。

In [ ]:
def show_video(split_dir, video_id, label="", n=12):
    """把某段影片均勻取樣 n 張影格畫成一列。"""
    folder = split_dir / str(video_id)
    frames = sorted(folder.glob("*.jpg"))
    if not frames:
        print(f"找不到影格: {folder}")
        return
    idx = np.linspace(0, len(frames) - 1, min(n, len(frames))).astype(int)
    fig, axes = plt.subplots(1, len(idx), figsize=(len(idx) * 1.5, 2.2))
    for ax, i in zip(np.atleast_1d(axes), idx):
        ax.imshow(Image.open(frames[i]))
        ax.set_title(f"#{i + 1}", fontsize=8)
        ax.axis("off")
    fig.suptitle(f"id={video_id}　label={label}　({len(frames)} frames)")
    plt.tight_layout()
    plt.show()

In [ ]:
# 隨機抽 4 段訓練影片（改 random_state 可換一批）
sample = train_df.sample(4, random_state=42)
for _, row in sample.iterrows():
    show_video(TRAIN_DIR, row["video_id"], row["label"])

## 6. 小結 —— 三個設計決策（已依實際數據填寫）

### 決策 1：取樣張數 `T`
- 全部影片**固定 37 幀**（Train / Validation 皆是），不需處理長度不一的問題。
- baseline 建議均勻取樣 **`T = 16`**；資源夠也可全取 37。
- 訓練時可抖動取樣起點當資料增強，驗證時固定均勻取樣。

### 決策 2：Resize 尺寸
- 影格高固定 100px；寬以 176px 為主（約 84%）、132px 次之（約 15%），少數其他。
- 寬度不一定要統一 → 直接 **resize 成 `112×112`**（torchvision 預訓練 3D CNN 的標準輸入；對邊緣裝置也夠輕）。

### 決策 3：類別不平衡
- 27 類大多 1,700–1,850 段；**最多** `Doing other things` ≈ 4,374、**最少** `Turning Hand Clockwise` ≈ 1,327，不平衡比約 **3.3**。
- 屬輕中度不平衡 → `CrossEntropyLoss` 帶 class weight 即可，先不必重取樣；評估務必看**混淆矩陣**。

> 評估集：**Test 無標籤**，練習階段以 **Validation 當評估集**算準確率 / 混淆矩陣。